In [1]:
# Cell 1: Imports and Setup
import os
import sys
# Ensure the root directory is in the path so we can import our package
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from kg_audit.entity.config_entity import TruthEvaluatorConfig, KGAuditPipelineConfig
from kg_audit.entity.artifact_entity import GraphRetrieverArtifact
from kg_audit.components.truth_evaluator import TruthEvaluator

pipeline_config = KGAuditPipelineConfig()
eval_config = TruthEvaluatorConfig(pipeline_config)
evaluator = TruthEvaluator(eval_config)

# Cell 2: Define the Ground Truth (What Neo4j holds)
# Reality: "The User (0) uses the Server (1). The Server (1) accesses the Database (2)."
gt_nodes = [
    {"id": 0, "name": "User"},
    {"id": 1, "name": "Server"},
    {"id": 2, "name": "Database"}
]
gt_edges = [
    {"source": 0, "target": 1, "type": "USES", "weight": 1.0},
    {"source": 1, "target": 2, "type": "ACCESSES", "weight": 1.0}
]

retriever_artifact = GraphRetrieverArtifact(
    query_entities=["User", "Database"],
    extracted_nodes=gt_nodes,
    extracted_edges=gt_edges,
    subgraph_file_path="mock_path.json",
    retrieval_status=True
)

# Cell 3: Define the LLM Response (The Hallucination)
# Hallucination: The LLM invents a direct link claiming "The User (0) accesses the Database (2) directly."
llm_nodes = gt_nodes.copy()
llm_edges = [
    {"source": 0, "target": 1, "type": "USES", "weight": 1.0},
    {"source": 1, "target": 2, "type": "ACCESSES", "weight": 1.0},
    {"source": 0, "target": 2, "type": "DIRECT_ACCESS", "weight": 1.0} # THE HALLUCINATION
]

# Cell 4: Run the CASSE Mathematical Engine
artifact = evaluator.evaluate_truth(
    retriever_artifact=retriever_artifact,
    response_nodes=llm_nodes,
    response_edges=llm_edges
)

print(f"Semantic Sheaf Energy (Inconsistency): {artifact.semantic_entailment_score:.4f}")
print(f"Curvature Distortion (Topological Holes): {artifact.triple_match_score:.4f}")
print(f"Spectral Divergence (Global Shift): {artifact.topological_similarity_score:.4f}")
print(f"\nFinal Accuracy Score: {artifact.final_accuracy_score:.4f}")
print(f"Hallucination Detected: {artifact.is_hallucinated}")

Semantic Sheaf Energy (Inconsistency): 2.2387
Curvature Distortion (Topological Holes): 2.0000
Spectral Divergence (Global Shift): 0.8382

Final Accuracy Score: 0.1552
Hallucination Detected: True
